# Multilingual-E5-Large Dense Ingestion into Qdrant Cloud

Ingests **corpus.csv** (261,597 rows) into the `vn_law_e5` Qdrant collection using **intfloat/multilingual-e5-large** — producing **dense** (1024-dim) vectors for semantic retrieval.

**Pipeline:**
1. Install / verify dependencies
2. Connect to Qdrant Cloud and create collection (idempotent)
3. Load & preview corpus
4. Chunk texts (≤ 400 words, sentence-boundary aware)
5. Encode with E5 (`"passage: "` prefix) and upsert in batches
6. Verify ingestion count

## 1. Install Dependencies

In [ ]:
# Run once to install required packages into the active venv
# (Skip if already installed)
%pip install -q qdrant-client python-dotenv pandas tqdm sentence-transformers
# Verify imports:
from sentence_transformers import SentenceTransformer
print("sentence-transformers import OK")

## 2. Imports & Configuration

In [ ]:
import json
import os
import re
import uuid

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

# ── Load credentials from backend/.env ──────────────────────────────────────
# Adjust the path if running from outside the project root
ENV_PATH = os.path.join("..", "backend", ".env")
load_dotenv(ENV_PATH, override=True)

QDRANT_URL     = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
_collections   = [c.strip() for c in os.getenv("COLLECTIONS", "vn_law_bge_m3,vn_law_e5").split(",")]
E5_COLLECTION  = _collections[1]

DENSE_DIM       = 1024   # multilingual-e5-large dense output dimension
BATCH_SIZE      = 64     # Adjust down if you run out of VRAM
CSV_PATH        = os.path.join("..", "data", "corpus.csv")
CHECKPOINT_FILE = os.path.join("..", "retrieval", "checkpoints", f"{E5_COLLECTION}.ckpt.json")

print(f"Qdrant URL      : {QDRANT_URL}")
print(f"E5 collection   : {E5_COLLECTION}")
print(f"CSV path        : {CSV_PATH}")
print(f"Checkpoint file : {CHECKPOINT_FILE}")

## 3. Connect to Qdrant Cloud & Create Collection

In [ ]:
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# List existing collections
existing = [c.name for c in client.get_collections().collections]
print("Existing collections:", existing)

if E5_COLLECTION in existing:
    print(f"\nCollection '{E5_COLLECTION}' already exists — skipping creation.")
    count = client.count(E5_COLLECTION).count
    print(f"Current point count: {count:,}")
else:
    client.create_collection(
        collection_name=E5_COLLECTION,
        vectors_config={
            "dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
        },
    )
    print(f"\nCreated collection '{E5_COLLECTION}' with dense ({DENSE_DIM}-dim) vectors.")

## 4. Load & Preview Corpus

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"\nNull counts:\n{df.isnull().sum()}")
df.head(3)

## 5. Define Chunking Helper

In [ ]:
def split_text_keeping_sentences(text: str, max_word_count: int = 400) -> list:
    """Split text into chunks of at most max_word_count words, preserving sentence boundaries."""
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks, current_chunk, current_wc = [], "", 0
    for sentence in sentences:
        wc = len(sentence.split())
        if current_wc + wc > max_word_count and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk, current_wc = sentence, wc
        else:
            current_chunk += (" " + sentence.strip()) if current_chunk else sentence.strip()
            current_wc += wc
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

# Quick sanity check
sample = df.iloc[0]["text"]
chunks = split_text_keeping_sentences(sample, max_word_count=400)
print(f"Sample row word count : {len(sample.split())}")
print(f"Chunks produced       : {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c.split())} words")

## 6. Chunk All Documents

In [ ]:
all_chunks = []   # list of (text, infor_id, chunk_id)
chunk_id = 0

for row in tqdm(df.itertuples(index=False), total=len(df), desc="Chunking"):
    text, cid = row.text, row.cid
    if not isinstance(text, str) or not text.strip():
        continue
    for chunk_text in split_text_keeping_sentences(text, max_word_count=400):
        all_chunks.append((chunk_text, int(cid), chunk_id))
        chunk_id += 1

print(f"Total chunks: {len(all_chunks):,}")

## 7. Load Multilingual-E5-Large Model

In [ ]:
# Downloads ~2GB from HuggingFace on first run; cached afterwards.
# Change "cuda:0" to "cpu" if no GPU is available (much slower).
model = SentenceTransformer("intfloat/multilingual-e5-large", device="cuda:0")
print("Multilingual-E5-Large loaded.")

## 7b. Checkpoint Helpers & Resume Detection

The checkpoint file records how many chunks have been successfully upserted.  
If the kernel crashes or the session is interrupted, rerun **all cells from the top** and the encoding loop will automatically skip the already-ingested chunks.

In [ ]:
def save_checkpoint(chunks_done: int, total_chunks: int) -> None:
    """Persist ingestion progress so the session can be resumed if interrupted."""
    os.makedirs(os.path.dirname(os.path.abspath(CHECKPOINT_FILE)), exist_ok=True)
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"chunks_done": chunks_done, "total_chunks": total_chunks}, f)


def load_checkpoint() -> int:
    """Return the chunk index to start from (0 if no checkpoint exists)."""
    if not os.path.exists(CHECKPOINT_FILE):
        return 0
    with open(CHECKPOINT_FILE) as f:
        data = json.load(f)
    print(f"[Checkpoint] Resuming from chunk {data['chunks_done']:,} / {data['total_chunks']:,}")
    return data["chunks_done"]


START_FROM = load_checkpoint()
if START_FROM > 0:
    print(f"Skipping first {START_FROM:,} already-ingested chunks.")
else:
    print("No checkpoint found — starting fresh.")

## 8. Encode with E5 & Upsert to Qdrant

E5 requires a `"passage: "` prefix for corpus documents (queries use `"query: "`).
Only dense vectors are produced and stored in Qdrant under the `"dense"` named vector slot.

In [ ]:
if START_FROM >= len(all_chunks):
    print(f"All {len(all_chunks):,} chunks already ingested — nothing to do.")
else:
    for i in tqdm(range(START_FROM, len(all_chunks), BATCH_SIZE), desc="Encoding & upserting"):
        batch = all_chunks[i : i + BATCH_SIZE]
        # E5 uses "passage: " prefix for corpus documents
        texts = ["passage: " + c[0] for c in batch]

        dense_vecs = model.encode(texts, normalize_embeddings=True)

        points = []
        for j, (text, infor_id, c_id) in enumerate(batch):
            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector={"dense": dense_vecs[j].tolist()},
                    payload={"text": text, "infor_id": infor_id, "chunk_id": c_id},
                )
            )

        client.upsert(collection_name=E5_COLLECTION, points=points)
        save_checkpoint(i + len(batch), len(all_chunks))

    print(f"\nEncoding & upsert complete — {len(all_chunks):,} total points.")

## 9. Verify Ingestion

In [ ]:
final_count = client.count(E5_COLLECTION).count
print(f"Points in '{E5_COLLECTION}': {final_count:,} / {len(all_chunks):,} expected")

# Clear checkpoint on successful completion
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print("Checkpoint cleared.")

# Fetch a sample point to confirm payload + vector shapes
sample_pts = client.scroll(E5_COLLECTION, limit=1, with_vectors=True, with_payload=True)[0]
if sample_pts:
    pt = sample_pts[0]
    print(f"\nSample point id     : {pt.id}")
    print(f"Payload keys        : {list(pt.payload.keys())}")
    print(f"Dense vector length : {len(pt.vector['dense'])}")
    print(f"Text preview        : {pt.payload['text'][:120]}...")